In [2]:
"""
╔══════════════════════════════════════════════════════════════════╗
║        ENTRENAMIENTO - Detector de Latas Multi-tarea            ║
║        Marca + Orientación con EfficientNetV2B0                 ║
╠══════════════════════════════════════════════════════════════════╣
║  Estructura dataset esperada:                                    ║
║    dataset\                                                      ║
║      aquarius-back\                                              ║
║      aquarius-front\                                             ║
║      cocacola-left\                                              ║
║      redbull-right\                                              ║
║      sprite-back\   monster-front\   titanium-right\  ...       ║
╚══════════════════════════════════════════════════════════════════╝
"""

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
DATASET_PATH = "dataset"
EPOCHS_FROZEN = 15
EPOCHS_FINE   = 10
OUTPUT_MODEL  = "modelo_latas.h5"
LABELS_FILE   = "labels.json"

# Marcas y orientaciones válidas (nombres completos tal como aparecen en las carpetas)
VALID_BRANDS     = {"titanium", "redbull", "monster", "cocacola", "sprite", "aquarius"}
VALID_ORIENTS    = {"front", "back", "left", "right"}

# ──────────────────────────────────────────────
# 1. PARSEAR DATASET
# ──────────────────────────────────────────────
def parse_dataset(dataset_path):
    """
    Lee carpetas con formato {marca}-{orientacion}
    Ejemplo: aquarius-back, cocacola-front, redbull-left
    """
    image_paths, brand_ids, orient_ids = [], [], []

    for folder in sorted(os.listdir(dataset_path)):
        folder_path = os.path.join(dataset_path, folder)
        if not os.path.isdir(folder_path):
            continue

        folder_lower = folder.strip().lower()

        # Separar por guión: "aquarius-back" → brand="aquarius", orient="back"
        if "-" not in folder_lower:
            print(f"⚠️  Carpeta ignorada: '{folder}' (sin guión, esperado ej: 'aquarius-back')")
            continue

        parts = folder_lower.split("-", 1)   # split en el primer guión
        brand, orient = parts[0], parts[1]

        if brand not in VALID_BRANDS:
            print(f"⚠️  Marca desconocida en '{folder}': '{brand}'. Válidas: {VALID_BRANDS}")
            continue
        if orient not in VALID_ORIENTS:
            print(f"⚠️  Orientación desconocida en '{folder}': '{orient}'. Válidas: {VALID_ORIENTS}")
            continue

        count = 0
        for fname in os.listdir(folder_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                image_paths.append(os.path.join(folder_path, fname))
                brand_ids.append(brand)
                orient_ids.append(orient)
                count += 1

        print(f"   📁 {folder_lower:<20s}  ({count} imágenes)")

    return image_paths, brand_ids, orient_ids


print("\n📂 Leyendo dataset...")
image_paths, brand_ids, orient_ids = parse_dataset(DATASET_PATH)
print(f"\n✅ Total imágenes encontradas: {len(image_paths)}")

if len(image_paths) == 0:
    raise ValueError("❌ No se encontraron imágenes. Revisa la ruta del dataset.")

# Convertir IDs a índices numéricos
unique_brands  = sorted(set(brand_ids))
unique_orients = sorted(set(orient_ids))

brand_to_idx  = {b: i for i, b in enumerate(unique_brands)}
orient_to_idx = {o: i for i, o in enumerate(unique_orients)}

brand_encoded  = np.array([brand_to_idx[b]  for b in brand_ids],  dtype=np.int32)
orient_encoded = np.array([orient_to_idx[o] for o in orient_ids], dtype=np.int32)

NUM_BRANDS     = len(unique_brands)
NUM_ORIENTATIONS = len(unique_orients)

print(f"\n   Marcas ({NUM_BRANDS}):        {unique_brands}")
print(f"   Orientaciones ({NUM_ORIENTATIONS}):  {unique_orients}")

# Guardar labels para inferencia posterior
# brands/orientations: {str(idx): nombre}  → usado en webcam_detector.py
labels_data = {
    "brands":        {str(i): b for i, b in enumerate(unique_brands)},
    "orientations":  {str(i): o for i, o in enumerate(unique_orients)},
    "brand_to_idx":  brand_to_idx,
    "orient_to_idx": orient_to_idx,
}
with open(LABELS_FILE, "w") as f:
    json.dump(labels_data, f, indent=2)
print(f"\n💾 Labels guardados en: {LABELS_FILE}")

# ──────────────────────────────────────────────
# 2. SPLIT TRAIN / VAL
# ──────────────────────────────────────────────
paths_arr = np.array(image_paths)

(train_paths, val_paths,
 train_brands, val_brands,
 train_orients, val_orients) = train_test_split(
    paths_arr, brand_encoded, orient_encoded,
    test_size=0.2, random_state=42, stratify=brand_encoded
)

print(f"\n   Train: {len(train_paths)} | Val: {len(val_paths)}")

# ──────────────────────────────────────────────
# 3. PIPELINE TF.DATA
# ──────────────────────────────────────────────
def load_image(path, brand, orient):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, {"brand_out": brand, "orient_out": orient}

def augment(img, labels):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.75, upper=1.25)
    img = tf.image.random_hue(img, max_delta=0.05)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, labels

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_brands, train_orients))
    .map(load_image,  num_parallel_calls=AUTOTUNE)
    .map(augment,     num_parallel_calls=AUTOTUNE)
    .shuffle(512)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_brands, val_orients))
    .map(load_image,  num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ──────────────────────────────────────────────
# 4. MODELO MULTI-TAREA (EfficientNetV2B0)
# ──────────────────────────────────────────────
print("\n🏗️  Construyendo modelo multi-tarea...")

base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))

# EfficientNetV2 hace su propio preprocesado internamente (rescaling 0-255)
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

# Trunk compartido
trunk = layers.Dense(256, activation="relu")(x)
trunk = layers.Dropout(0.4)(trunk)

# ── Cabeza 1: Marca ──────────────────────────
brand_head = layers.Dense(128, activation="relu", name="brand_dense")(trunk)
brand_head = layers.Dropout(0.3)(brand_head)
brand_out  = layers.Dense(NUM_BRANDS, activation="softmax", name="brand_out")(brand_head)

# ── Cabeza 2: Orientación ────────────────────
orient_head = layers.Dense(64, activation="relu", name="orient_dense")(trunk)
orient_head = layers.Dropout(0.3)(orient_head)
orient_out  = layers.Dense(NUM_ORIENTATIONS, activation="softmax", name="orient_out")(orient_head)

model = keras.Model(inputs=inputs, outputs=[brand_out, orient_out])

# ──────────────────────────────────────────────
# 5. COMPILAR (fase 1 - base congelada)
# ──────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={
        "brand_out":  "sparse_categorical_crossentropy",
        "orient_out": "sparse_categorical_crossentropy",
    },
    loss_weights={"brand_out": 1.0, "orient_out": 0.8},
    metrics={
        "brand_out":  ["accuracy"],
        "orient_out": ["accuracy"],
    }
)

model.summary()

# ──────────────────────────────────────────────
# 6. ENTRENAMIENTO FASE 1 (cabezas)
# ──────────────────────────────────────────────
print(f"\n🚀 Fase 1: Entrenando cabezas ({EPOCHS_FROZEN} épocas)...")

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor="val_loss"),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=callbacks_phase1,
)

# ──────────────────────────────────────────────
# 7. FINE-TUNING (descongelar últimas 30 capas)
# ──────────────────────────────────────────────
print("\n🔧 Fase 2: Fine-tuning...")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss={
        "brand_out":  "sparse_categorical_crossentropy",
        "orient_out": "sparse_categorical_crossentropy",
    },
    loss_weights={"brand_out": 1.0, "orient_out": 0.8},
    metrics={
        "brand_out":  ["accuracy"],
        "orient_out": ["accuracy"],
    }
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
    keras.callbacks.ModelCheckpoint(OUTPUT_MODEL, save_best_only=True, monitor="val_loss"),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks_phase2,
)

# ──────────────────────────────────────────────
# 8. GUARDAR MODELO
# ──────────────────────────────────────────────
model.save(OUTPUT_MODEL)
print(f"\n✅ Modelo guardado: {OUTPUT_MODEL}")

# ──────────────────────────────────────────────
# 9. GRÁFICAS
# ──────────────────────────────────────────────
def plot_histories(h1, h2):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Entrenamiento - Detector de Latas", fontsize=14, fontweight="bold")

    for ax, key, title in zip(
        axes,
        ["brand_out_accuracy", "orient_out_accuracy"],
        ["Accuracy - Marca", "Accuracy - Orientación"]
    ):
        vals     = h1.history[key]     + h2.history[key]
        val_vals = h1.history[f"val_{key}"] + h2.history[f"val_{key}"]
        split    = len(h1.history[key])

        ax.plot(vals,     label="Train")
        ax.plot(val_vals, label="Val")
        ax.axvline(x=split - 1, color="gray", linestyle="--", label="Fine-tune →")
        ax.set_title(title)
        ax.set_xlabel("Época")
        ax.set_ylabel("Accuracy")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.show()
    print("📊 Gráfica guardada: training_history.png")

plot_histories(history1, history2)


📂 Leyendo dataset...
   📁 aquarius-back         (200 imágenes)
   📁 aquarius-front        (200 imágenes)
   📁 aquarius-left         (200 imágenes)
   📁 aquarius-right        (200 imágenes)
   📁 cocacola-back         (200 imágenes)
   📁 cocacola-front        (200 imágenes)
   📁 cocacola-left         (200 imágenes)
   📁 cocacola-right        (200 imágenes)
   📁 monster-back          (200 imágenes)
   📁 monster-front         (200 imágenes)
   📁 monster-left          (200 imágenes)
   📁 monster-right         (200 imágenes)
   📁 redbull-back          (200 imágenes)
   📁 redbull-front         (200 imágenes)
   📁 redbull-left          (200 imágenes)
   📁 redbull-right         (200 imágenes)
   📁 sprite-back           (200 imágenes)
   📁 sprite-front          (200 imágenes)
   📁 sprite-left           (200 imágenes)
   📁 sprite-right          (200 imágenes)
   📁 titanium-back         (200 imágenes)
   📁 titanium-front        (200 imágenes)
   📁 titanium-left         (200 imágenes)
   📁 titaniu

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-b0   │ (None, 7, 7,      │  5,919,312 │ input_layer_1[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetv2-b… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    327,936 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ brand_dense (Dense) │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ orient_dense        │ (None, 64)        │     16,448 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ brand_dense[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ orient_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ brand_out (Dense)   │ (None, 6)         │        774 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ orient_out (Dense)  │ (None, 4)         │        260 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,302,746 (24.04 MB)

 Trainable params: 380,874 (1.45 MB)

 Non-trainable params: 5,921,872 (22.59 MB)


🚀 Fase 1: Entrenando cabezas (15 épocas)...
Epoch 1/15


KeyboardInterrupt: 

In [2]:
"""
╔══════════════════════════════════════════════════════════════════╗
║        DETECCIÓN EN TIEMPO REAL - Webcam                        ║
║        Usa el modelo entrenado para detectar marca+orientación  ║
╚══════════════════════════════════════════════════════════════════╝

Uso:
    python webcam_detector.py
    python webcam_detector.py --camera 1   # si tienes múltiples cámaras
    python webcam_detector.py --threshold 0.7
"""

import cv2
import json
import numpy as np
import tensorflow as tf
import time

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
MODEL_PATH   = "modelo_latas.h5"
LABELS_FILE  = "labels.json"
IMG_SIZE     = (224, 224)
CONF_THRESHOLD = 0.6   # confianza mínima para mostrar resultado

# Colores por orientación (BGR)
ORIENT_COLORS = {
    "front": (0,   200,  50),   # verde
    "back":  (50,  50,  220),   # rojo
    "left":  (220, 150,   0),   # azul
    "right": (0,  180,  220),   # amarillo
}

# Icono de orientación
ORIENT_ICON = {
    "front": "⬛ FRONT",
    "back":  "⬛ BACK ",
    "left":  "◀  LEFT ",
    "right": "▶  RIGHT",
}

# ──────────────────────────────────────────────
# CARGAR MODELO Y ETIQUETAS
# ──────────────────────────────────────────────
def load_model_and_labels():
    print("⏳ Cargando modelo...")
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f"✅ Modelo cargado: {MODEL_PATH}")

    with open(LABELS_FILE, "r") as f:
        labels = json.load(f)

    brands       = labels["brands"]        # {idx: nombre}
    orientations = labels["orientations"]  # {idx: nombre}

    print(f"   Marcas: {list(brands.values())}")
    print(f"   Orientaciones: {list(orientations.values())}\n")

    return model, brands, orientations


# ──────────────────────────────────────────────
# PREPROCESAR FRAME PARA EL MODELO
# ──────────────────────────────────────────────
def preprocess_frame(frame):
    """
    Recorta el ROI central del frame (cuadrado) y lo prepara para el modelo.
    """
    h, w = frame.shape[:2]
    side = min(h, w)
    y0   = (h - side) // 2
    x0   = (w - side) // 2
    roi  = frame[y0:y0+side, x0:x0+side]

    img = cv2.resize(roi, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32)
    img = np.expand_dims(img, axis=0)  # (1, 224, 224, 3)
    return img, (x0, y0, side)


# ──────────────────────────────────────────────
# OVERLAY EN EL FRAME
# ──────────────────────────────────────────────
def draw_overlay(frame, brand_name, brand_conf, orient_name, orient_conf, roi_box, fps):
    x0, y0, side = roi_box
    h, w = frame.shape[:2]

    # ── ROI box ─────────────────────────────
    color = ORIENT_COLORS.get(orient_name, (200, 200, 200))
    cv2.rectangle(frame, (x0, y0), (x0+side, y0+side), color, 2)

    # ── Panel de info (parte inferior) ───────
    panel_h = 90
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, h - panel_h), (w, h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)

    font       = cv2.FONT_HERSHEY_DUPLEX
    font_small = cv2.FONT_HERSHEY_SIMPLEX

    # Marca
    brand_text = f"MARCA:  {brand_name.upper()}"
    conf_b_txt = f"{brand_conf*100:.1f}%"
    cv2.putText(frame, brand_text, (15, h - panel_h + 28), font, 0.75, (255,255,255), 1)
    cv2.putText(frame, conf_b_txt, (w - 75, h - panel_h + 28), font, 0.65, (120,255,120), 1)

    # Orientación
    orient_text = f"ORIENT: {orient_name.upper()}"
    conf_o_txt  = f"{orient_conf*100:.1f}%"
    cv2.putText(frame, orient_text, (15, h - panel_h + 60), font, 0.75, color, 1)
    cv2.putText(frame, conf_o_txt, (w - 75, h - panel_h + 60), font, 0.65, (120,255,120), 1)

    # FPS
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 100, 25), font_small, 0.55, (180,180,180), 1)

    # Barra de confianza marca
    bar_w = int((w - 30) * brand_conf)
    cv2.rectangle(frame, (15, h - 8), (w - 15, h - 3), (50,50,50), -1)
    cv2.rectangle(frame, (15, h - 8), (15 + bar_w, h - 3), (120,255,120), -1)

    return frame


def draw_low_conf(frame, fps):
    """Mostrar cuando la confianza es baja."""
    h, w = frame.shape[:2]
    cv2.putText(frame, "Acerca una lata...", (w//2 - 130, h//2),
                cv2.FONT_HERSHEY_DUPLEX, 0.8, (100,100,255), 1)
    cv2.putText(frame, f"FPS: {fps:.1f}", (w-100, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180,180,180), 1)


# ──────────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────────
def main(camera_id: int, threshold: float):
    model, brands, orientations = load_model_and_labels()

    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        raise RuntimeError(f"❌ No se pudo abrir la cámara {camera_id}")

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cap.set(cv2.CAP_PROP_FPS, 30)

    print("🎥 Webcam iniciada. Presiona Q para salir.\n")

    prev_time = time.time()
    fps       = 0.0

    # Suavizado (promedio móvil de las últimas N predicciones)
    SMOOTH_N   = 5
    brand_buf  = []
    orient_buf = []

    while True:
        ret, frame = cap.read()
        if not ret:
            print("⚠️  No se pudo leer frame.")
            break

        # FPS
        now       = time.time()
        fps       = 0.9 * fps + 0.1 * (1.0 / max(now - prev_time, 1e-6))
        prev_time = now

        # Preprocesar
        img_input, roi_box = preprocess_frame(frame)

        # Predicción
        brand_probs, orient_probs = model.predict(img_input, verbose=0)
        brand_probs  = brand_probs[0]   # (N_brands,)
        orient_probs = orient_probs[0]  # (N_orients,)

        brand_idx  = int(np.argmax(brand_probs))
        orient_idx = int(np.argmax(orient_probs))
        brand_conf  = float(brand_probs[brand_idx])
        orient_conf = float(orient_probs[orient_idx])

        # Suavizado temporal
        brand_buf.append(brand_idx)
        orient_buf.append(orient_idx)
        if len(brand_buf)  > SMOOTH_N: brand_buf.pop(0)
        if len(orient_buf) > SMOOTH_N: orient_buf.pop(0)

        smooth_brand  = max(set(brand_buf),  key=brand_buf.count)
        smooth_orient = max(set(orient_buf), key=orient_buf.count)

        brand_name  = brands[str(smooth_brand)]
        orient_name = orientations[str(smooth_orient)]

        # Dibujar
        if brand_conf >= threshold:
            frame = draw_overlay(
                frame,
                brand_name, brand_conf,
                orient_name, orient_conf,
                roi_box, fps
            )
        else:
            # Dibujar ROI de todas formas
            x0, y0, side = roi_box
            cv2.rectangle(frame, (x0, y0), (x0+side, y0+side), (80,80,80), 1)
            draw_low_conf(frame, fps)

        cv2.imshow("🥤 Detector de Latas | Q para salir", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q") or key == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    print("👋 Detector cerrado.")


# ──────────────────────────────────────────────
# CONFIGURACIÓN MANUAL (cambia estos valores si necesitas)
# Compatible con Jupyter, VSCode, terminal — en cualquier entorno
# ──────────────────────────────────────────────
CAMERA_ID  =0      # 0 = cámara principal, 1 = secundaria, etc.
THRESHOLD  = CONF_THRESHOLD  # confianza mínima (0.0 - 1.0)

if __name__ == "__main__":
    main(CAMERA_ID, THRESHOLD)

⏳ Cargando modelo...


✅ Modelo cargado: modelo_latas.h5
   Marcas: ['aquarius', 'cocacola', 'monster', 'redbull', 'sprite', 'titanium']
   Orientaciones: ['back', 'front', 'left', 'right']

🎥 Webcam iniciada. Presiona Q para salir.

👋 Detector cerrado.
